In [6]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Định nghĩa đường dẫn file
file_path = '../data/dulieuketoan.csv'

# 2. Đọc file dữ liệu
try:
    df = pd.read_csv(file_path)
    print(f" Đọc dữ liệu thành công! Kích thước: {df.shape[0]} dòng, {df.shape[1]} cột.")
except FileNotFoundError:
    print(" Lỗi: Không tìm thấy file dữ liệu. Hãy kiểm tra lại folder 'data' nhé!")

 Đọc dữ liệu thành công! Kích thước: 6819 dòng, 96 cột.


In [7]:
# Kiểm tra tên các cột ban đầu
print("Tên các cột trước khi sửa:", df.columns.tolist()[:5]) # In thử 5 cột đầu

# Thực hiện xóa khoảng trắng thừa ở đầu và cuối tên cột
df.columns = df.columns.str.strip()

print("\n Tên các cột sau khi chuẩn hóa:", df.columns.tolist()[:5])

Tên các cột trước khi sửa: ['Bankrupt?', ' ROA(C) before interest and depreciation before interest', ' ROA(A) before interest and % after tax', ' ROA(B) before interest and depreciation after tax', ' Operating Gross Margin']

 Tên các cột sau khi chuẩn hóa: ['Bankrupt?', 'ROA(C) before interest and depreciation before interest', 'ROA(A) before interest and % after tax', 'ROA(B) before interest and depreciation after tax', 'Operating Gross Margin']


In [8]:
# 1. Tính tổng số lượng giá trị trống ở mỗi cột
missing_values = df.isnull().sum()
print("Số lượng dòng trống ở từng cột:")
print(missing_values[missing_values > 0]) # Chỉ in những cột có giá trị trống

# 2. Xử lý giá trị trống
# Cách an toàn nhất cho dữ liệu tài chính lớn: Xóa các dòng bị khuyết thông tin quan trọng
df_cleaned = df.dropna()

print(f"\n Dữ liệu sau khi xóa bỏ các dòng trống: {df_cleaned.shape[0]} dòng.")

Số lượng dòng trống ở từng cột:
Series([], dtype: int64)

 Dữ liệu sau khi xóa bỏ các dòng trống: 6819 dòng.


In [9]:
# 1. Đếm số dòng trùng lặp hoàn toàn
duplicate_count = df_cleaned.duplicated().sum()
print(f"Số dòng dữ liệu bị trùng lặp: {duplicate_count}")

# 2. Nếu có trùng lặp thì tiến hành xóa
if duplicate_count > 0:
    df_cleaned = df_cleaned.drop_duplicates()
    print(f" Đã xóa các dòng trùng! Kích thước mới: {df_cleaned.shape[0]} dòng.")
else:
    print(" Dữ liệu sạch sẽ, không bị trùng lặp!")

Số dòng dữ liệu bị trùng lặp: 0
 Dữ liệu sạch sẽ, không bị trùng lặp!


In [11]:
# Tên cột mục tiêu phân loại trạng thái doanh nghiệp (Phá sản hay An toàn)
target_column = 'Bankrupt?' 

# 1. Ép kiểu dữ liệu của cột mục tiêu về dạng số nguyên (0 hoặc 1) để tránh lỗi 'unknown'
df_cleaned[target_column] = df_cleaned[target_column].astype(int)

# 2. Tách đặc trưng (X) và nhãn mục tiêu (y)
X = df_cleaned.drop(columns=[target_column])
y = df_cleaned[target_column]

# 3. Chia dữ liệu theo tỷ lệ 80% Train - 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f" Kích thước tập Train (X_train): {X_train.shape}")
print(f" Kích thước tập Test (X_test): {X_test.shape}")

 Kích thước tập Train (X_train): (5455, 95)
 Kích thước tập Test (X_test): (1364, 95)


In [12]:
# Khởi tạo bộ chuẩn hóa
scaler = StandardScaler()

# Chỉ chuẩn hóa trên các cột số (loại trừ các cột định danh như Mã doanh nghiệp, Tên công ty nếu có)
# Tự động lấy các cột có kiểu dữ liệu là số
numeric_cols = X_train.select_dtypes(include=[np.number]).columns

# Tiến hành fit và transform trên tập Train, sau đó áp dụng (transform) lên tập Test
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(" Đã chuẩn hóa StandardScaler xong cho các biến tài chính!")
# Xem thử dữ liệu biến đổi của dòng đầu tiên
print(X_train_scaled[numeric_cols].iloc[0][:5])

 Đã chuẩn hóa StandardScaler xong cho các biến tài chính!
ROA(C) before interest and depreciation before interest   -0.176677
ROA(A) before interest and % after tax                    -0.117303
ROA(B) before interest and depreciation after tax         -0.135577
Operating Gross Margin                                    -0.542117
Realized Sales Gross Margin                               -0.541389
Name: 318, dtype: float64


In [14]:
# Gộp lại nhãn mục tiêu vào tập dữ liệu đã chuẩn hóa để xuất ra file hoàn chỉnh
train_final = X_train_scaled.copy()
train_final[target_column] = y_train

test_final = X_test_scaled.copy()
test_final[target_column] = y_test

# Tạo file tổng hợp sau làm sạch
df_final_clean = pd.concat([train_final, test_final], axis=0)

# Lưu lại vào folder data
output_path = '../data/dulieuketoan_clean.csv'
df_final_clean.to_csv(output_path, index=False)

print(f" Xuất file dữ liệu sạch thành công tại: {output_path}")


 Xuất file dữ liệu sạch thành công tại: ../data/dulieuketoan_clean.csv
